In [6]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [8]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [10]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [11]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [12]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course late — can I still join, or is it too late to start?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late to join course start too late enroll late found course late can I still join"}', call_id='call_AjMO4cT0Hrb7d5c6YIN81dP2', name='search', type='function_call', id='fc_0b6237fca0431bc9006a52fcada1a881a38c5888854eccb1d0', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_AjMO4cT0Hrb7d5c6YIN81dP2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you n

In [14]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [15]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late to join course start too late enroll late found course late can I still join"}'}]

In [16]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [17]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course late — can I still join, or is it too late to start?',
 'answer_agent': 'Yes — you can still join.\n\nAccording to the course FAQ, if you just discovered the course, you can start anytime. The only caveat is that if you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also point you to the recommended place to start.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late to join course start too late enroll late found course late can I still join"}'}],
 'cost': Decimal('0.0011925'),
 'document': '74eb249bbf'}

In [18]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [20]:
df_agent = pd.DataFrame(agent_answers)
df_agent.head(5)

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course late — can I still jo...,Yes — you can still join. The course materials...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.0011160,74eb249bbf
1,Am I allowed to enroll if I only discovered th...,"Yes — if you just discovered the course, you c...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.00123225,74eb249bbf
2,"If I join after the course has started, can I ...",Yes — if you join after the course has started...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.0010215,74eb249bbf
3,What do I need to do to qualify for the certif...,"Yes — if you’re joining late, you can still qu...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00129975,74eb249bbf
4,"Is it okay to start the course now, and what’s...",Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""st...",0.00224775,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.06514800')

In [22]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [23]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [25]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [26]:
import ast
import json
from evaluation_utils import calc_total_price, llm_structured_retry


def _parse_tool_calls(tool_calls):
    if isinstance(tool_calls, list):
        return tool_calls

    if not tool_calls:
        return []

    if isinstance(tool_calls, str):
        raw = tool_calls.strip()
        if not raw:
            return []

        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(raw)
            except (ValueError, SyntaxError):
                return raw

    return tool_calls


def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = _parse_tool_calls(rec["tool_calls"])

    if isinstance(tool_calls, (list, dict)):
        tool_calls_text = json.dumps(tool_calls, indent=2)
    else:
        tool_calls_text = str(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=tool_calls_text,
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [27]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It clearly says the student can still join and adds the key condition that to receive a certificate, the project must be submitted while submissions are still open. This preserves the essential information from the original answer, with some extra wording about materials being available anytime, which does not conflict with the ground truth.', answer_score='good', trajectory_reasoning='The search query is relevant to the student’s question and includes the key idea of joining late / too late to start. Only one search call was made, which is reasonable for a straightforward FAQ-style question. There were no unnecessary duplicate calls, and the tool use supports the final answer.', trajectory_score='good')

In [28]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [30]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [32]:
df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval.head(5)

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course late — can I still jo...,74eb249bbf,good,The agent answer matches the ground truth. It ...,good,The search query was relevant to the question ...
1,Am I allowed to enroll if I only discovered th...,74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query was relevant to the user’s qu...
2,"If I join after the course has started, can I ...",74eb249bbf,good,The agent’s answer matches the ground truth on...,good,The search query was relevant and contained th...
3,What do I need to do to qualify for the certif...,74eb249bbf,good,The agent’s answer matches the ground truth on...,good,The search query is relevant to the question a...
4,"Is it okay to start the course now, and what’s...",74eb249bbf,good,The agent’s answer is mostly aligned with the ...,good,The search queries were relevant to the user’s...


In [33]:
calc_total_price(usages)

0.05443350000000001

In [34]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    48
bad      2
Name: count, dtype: int64

In [35]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    49
bad      1
Name: count, dtype: int64

In [36]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)